# MSMARCO-XI -> Jina v3 -> Qdrant Cloud ingestion

Implements `claude/ingestion.md`. Runs top to bottom in Colab (T4 GPU).

**This notebook has a hard-stop gate after Phase 0.** Run the Phase 0 cell,
read the printed report, fill in `FIELD_MAP` / tier language codes in the
cell right after it, then continue. Do not run the rest blind.

See `data_ingestion/guide.md` for setup instructions.


## 0. Setup

In [ ]:
!pip install -q "qdrant-client[fastembed]" datasets transformers sentencepiece einops torch numpy tqdm huggingface_hub


In [ ]:
import os, sys, json, time, uuid, re, hashlib, itertools
from dataclasses import dataclass
import numpy as np
import torch
from tqdm.auto import tqdm

# Try to mount Drive (checkpoints/artifacts must survive Colab disconnects).
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Not running in Colab (or Drive mount failed) — continuing without it:", e)

# Locate config.py: works whether you `git clone`d the repo (preferred) or
# uploaded data_ingestion/config.py + this notebook side by side.
for p in ["/content/hhg-voice-rag-system/data_ingestion", "/content/data_ingestion", "/content", "."]:
    if os.path.isdir(p) and os.path.exists(os.path.join(p, "config.py")):
        sys.path.insert(0, p)
        break

from config import CFG
print("Loaded config. QDRANT_URL set:", bool(CFG.QDRANT_URL), "| QDRANT_API_KEY set:", bool(CFG.QDRANT_API_KEY))
os.makedirs(CFG.DRIVE_DIR, exist_ok=True)


## 1. Phase 0 — schema discovery (HARD STOP GATE)

Do not run anything below this section until you have read the printed
report and confirmed it in the next cell. The dataset card may be
inconsistent about field names — this discovers them instead of assuming.


In [ ]:
from datasets import get_dataset_config_names, load_dataset
from huggingface_hub import login as hf_login

def run_phase0_discovery(cfg):
    report = {"configs": [], "gated": False, "samples": {}}
    try:
        configs = get_dataset_config_names(cfg.DATASET_NAME)
    except Exception as e:
        msg = str(e)
        if "gated" in msg.lower() or "authentication" in msg.lower() or "401" in msg:
            report["gated"] = True
            if not cfg.HF_TOKEN:
                print("Dataset appears GATED and no HF_TOKEN is set in config.py. Set it and re-run.")
                return report
            hf_login(token=cfg.HF_TOKEN)
            configs = get_dataset_config_names(cfg.DATASET_NAME, token=cfg.HF_TOKEN)
        else:
            raise
    report["configs"] = configs
    print("CONFIGS:", configs)

    for c in configs[:5]:
        try:
            ds = load_dataset(cfg.DATASET_NAME, c, split="train", streaming=True,
                               token=(cfg.HF_TOKEN or None))
            row = next(iter(ds))
            keys = list(row.keys())
            sample = {k: str(v)[:200] for k, v in row.items()}
            report["samples"][c] = {"keys": keys, "sample": sample}
            print(f"\n[{c}] KEYS:", keys)
            print(f"[{c}] SAMPLE:", sample)
        except Exception as e:
            print(f"[{c}] FAILED to sample:", e)

    print("""
---- Phase 0 report — answer these before continuing ----
1. Language config names -> map into TIERS below (§ next cell).
2. Gold-passage signal present? (is_selected / label / relevance / qrels / answers)
   If ABSENT: STOP. There is no Recall@10 ground truth — escalate before proceeding.
3. Passage container shape: one row per (query,passage) pair, or one row per
   query with a list of passages? -> set FIELD_MAP['passage_container'].
4. Are query_id / passage_id stable, or must they be synthesised?
5. Is the dataset gated? (see 'gated' above) -> HF_TOKEN required if so.
-----------------------------------------------------------
""")
    return report

PHASE0_REPORT = run_phase0_discovery(CFG)


### Fill this in from the Phase 0 report, then set `PHASE0_CONFIRMED = True`

`FIELD_MAP` is the adapter — the rest of the script never touches raw
dataset keys directly, only through `get_field()`. If a language config is
missing a field the others have, it is degraded to atomic-only, not crashed
(see §11 in the spec).


In [ ]:
FIELD_MAP = {
    "query_id":          "query_id",     # <-- edit from Phase 0 output
    "query":             "query",
    "passages":          "passages",     # list-of-text field, or the single passage text field
    "passage_id":        "passage_id",
    "is_gold":           "is_selected",  # or None if no gold signal (see warning below)
    "passage_container": "list",         # "list" = one row/query with N passages, "single" = one row per (query,passage)
}

def get_field(row, logical_key, default=None):
    real = FIELD_MAP.get(logical_key)
    if not real:
        return default
    return row.get(real, default)

if FIELD_MAP.get("is_gold") is None:
    print("WARNING: no gold-passage signal configured. Per spec §2 this should STOP the run — "
          "Recall@10 has no ground truth without it. Only continue if you have escalated and "
          "deliberately decided to proceed.")

# Flip this only after you've actually read the Phase 0 report above.
PHASE0_CONFIRMED = False
assert PHASE0_CONFIRMED, "Set PHASE0_CONFIRMED = True after reviewing the Phase 0 report and FIELD_MAP."
print("Phase 0 confirmed. Proceeding.")


## 2. Matryoshka truncation — truncate first, then normalize (§1)

In [ ]:
def matryoshka(v_full: np.ndarray, dim: int) -> np.ndarray:
    """v_full: (N, 1024) float32. Returns (N, dim) float32, L2-normalized."""
    v = v_full[:, :dim]                                   # 1. SLICE
    n = np.linalg.norm(v, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return (v / n).astype(np.float32)                     # 2. THEN NORMALIZE

# Sanity check (not a test suite — just the invariant the spec calls out).
_probe = np.random.randn(16, 1024).astype(np.float32)
_out = matryoshka(_probe, 256)
assert np.allclose(np.linalg.norm(_out, axis=1), 1.0, atol=1e-5)
print("matryoshka() normalization check passed")


## 3. Language tiers (§3)

In [ ]:
TIERS_SPEC = {
    "A": {"langs": ["en"], "queries_per_lang": 10_000, "distractors": 2,
          "strategies": ["atomic", "window", "enriched", "query"]},
    "B": {"langs": ["hi", "kn", "ta", "te", "ml"], "queries_per_lang": 6_000, "distractors": 2,
          "strategies": ["atomic", "window", "enriched", "query"]},
    "C": {"langs": ["mr", "or", "kok", "bn"], "queries_per_lang": 2_500, "distractors": 2,
          "strategies": ["atomic", "query"]},
    "D": {"langs": ["__REST__"], "queries_per_lang": 1_000, "distractors": 1,
          "strategies": ["atomic"]},
}

def _config_row_count(dataset_name, cfg_name, hf_token):
    try:
        from datasets import load_dataset_builder
        b = load_dataset_builder(dataset_name, cfg_name, token=(hf_token or None))
        return b.info.splits["train"].num_examples
    except Exception:
        return -1  # unknown; falls back to discovery order

def resolve_tiers(tiers_spec, discovered_configs, dataset_name, hf_token):
    resolved, claimed, warnings = {}, set(), []
    for tname in ["A", "B", "C"]:
        langs = []
        for l in tiers_spec[tname]["langs"]:
            if l in discovered_configs:
                langs.append(l); claimed.add(l)
            else:
                warnings.append(f"Tier {tname}: language '{l}' not in dataset configs — skipping, not substituting.")
        resolved[tname] = {**tiers_spec[tname], "langs": langs}

    rest = [c for c in discovered_configs if c not in claimed]
    counts = {c: _config_row_count(dataset_name, c, hf_token) for c in rest}
    rest_sorted = sorted(rest, key=lambda c: counts[c], reverse=True)
    d_langs = rest_sorted[:8]
    if len(rest_sorted) > 8:
        warnings.append(f"Tier D capped at 8 of {len(rest_sorted)} remaining languages; dropped: {rest_sorted[8:]}")
    resolved["D"] = {**tiers_spec["D"], "langs": d_langs}

    for w in warnings:
        print("WARNING:", w)
    return resolved, warnings

def project_point_count(resolved):
    total = 0
    for t in resolved.values():
        for _ in t["langs"]:
            q, d, strat = t["queries_per_lang"], t["distractors"], set(t["strategies"])
            n = q * (1 + d)                       # atomic
            if "window" in strat: n += q * (1 + d) * 0.25
            if "enriched" in strat: n += q
            if "query" in strat: n += q
            total += n
    return int(total)

def enforce_point_budget(resolved, max_points, warnings):
    total = project_point_count(resolved)
    if total <= max_points:
        return resolved, total
    print(f"Projected {total} points exceeds MAX_POINTS={max_points}; trimming tier D then tier C.")
    for tname in ["D", "C"]:
        if total <= max_points:
            break
        t = resolved[tname]
        if not t["langs"]:
            continue
        factor = max(0.1, (max_points - (total - project_point_count({tname: t}))) /
                     max(1, project_point_count({tname: t})))
        old_q = t["queries_per_lang"]
        t["queries_per_lang"] = max(100, int(old_q * factor))
        warnings.append(f"Tier {tname}: queries_per_lang {old_q} -> {t['queries_per_lang']} to stay under budget.")
        total = project_point_count(resolved)
    print(f"Adjusted projected total: {total}")
    return resolved, total

TIERS_RESOLVED, _tier_warnings = resolve_tiers(TIERS_SPEC, PHASE0_REPORT["configs"], CFG.DATASET_NAME, CFG.HF_TOKEN)
TIERS_RESOLVED, PROJECTED_POINTS = enforce_point_budget(TIERS_RESOLVED, CFG.MAX_POINTS, _tier_warnings)
print("Resolved tiers:", json.dumps({k: v["langs"] for k, v in TIERS_RESOLVED.items()}, ensure_ascii=False, indent=2))
print("Projected point count:", PROJECTED_POINTS)


## 4. Retrieval unit construction (§4)

In [ ]:
NS = uuid.UUID("6ba7b810-9dad-11d1-80b4-00c04fd430c8")

def point_id_for(uid: str) -> str:
    return str(uuid.uuid5(NS, uid))

SENT_SPLIT_RE = re.compile(r"[।॥.!?\n]+")

def split_sentences(text: str):
    parts = [p.strip() for p in SENT_SPLIT_RE.split(text) if p.strip()]
    if len(parts) >= 2:
        return parts
    toks = text.split()
    if not toks:
        return [text]
    return [" ".join(toks[i:i + 40]) for i in range(0, len(toks), 40)]

def make_windows(sentences, size=3, stride=2):
    windows, i = [], 0
    while i < len(sentences):
        w = sentences[i:i + size]
        if w:
            windows.append(" ".join(w))
        if i + size >= len(sentences):
            break
        i += stride
    return windows

def build_atomic_units(lang, tier, qid, gold, distractors):
    units = []
    for i, p in enumerate([gold] + distractors):
        uid = f"{lang}|atomic|{qid}|p{i:02d}"
        units.append({
            "uid": uid, "embed_text": p["text"],
            "payload": {"text": p["text"], "lang": lang, "tier": tier, "strategy": "atomic",
                        "qid": qid, "pid": p["pid"], "parent_id": None,
                        "query_text": None, "answer_text": None,
                        "is_gold": (i == 0), "n_chars": len(p["text"]), "src_row": p["src_row"]},
        })
    return units

def build_window_units(lang, tier, qid, gold, median_len):
    if len(gold["text"]) <= median_len:
        return []
    parent_id = point_id_for(f"{lang}|atomic|{qid}|p00")
    windows = make_windows(split_sentences(gold["text"]), size=3, stride=2)
    units = []
    for wi, wtext in enumerate(windows):
        uid = f"{lang}|window|{qid}|w{wi:02d}"
        units.append({
            "uid": uid, "embed_text": wtext,
            "payload": {"text": wtext, "lang": lang, "tier": tier, "strategy": "window",
                        "qid": qid, "pid": gold["pid"], "parent_id": parent_id,
                        "query_text": None, "answer_text": None,
                        "is_gold": True, "n_chars": len(wtext), "src_row": gold["src_row"]},
        })
    return units

def build_enriched_unit(lang, tier, qid, query_text, gold):
    uid = f"{lang}|enriched|{qid}|p00"
    return {"uid": uid, "embed_text": f"{query_text}\n\n{gold['text']}",
            "payload": {"text": gold["text"], "lang": lang, "tier": tier, "strategy": "enriched",
                        "qid": qid, "pid": gold["pid"], "parent_id": None,
                        "query_text": None, "answer_text": None,
                        "is_gold": True, "n_chars": len(gold["text"]), "src_row": gold["src_row"]}}

def build_query_unit(lang, tier, qid, query_text, gold):
    uid = f"{lang}|query|{qid}|q00"
    return {"uid": uid, "embed_text": query_text,
            "payload": {"text": gold["text"], "lang": lang, "tier": tier, "strategy": "query",
                        "qid": qid, "pid": gold["pid"], "parent_id": None,
                        "query_text": query_text, "answer_text": gold["text"],
                        "is_gold": True, "n_chars": len(query_text), "src_row": gold["src_row"]}}

def build_units_for_group(group, lang, tier, strategies, median_len, n_distractors):
    passages = group["passages"]
    if not passages:
        return []
    gold = next((p for p in passages if p.get("is_gold")), passages[0])
    distractors = [p for p in passages if p is not gold][:n_distractors]
    units = []
    if "atomic" in strategies:
        units += build_atomic_units(lang, tier, group["qid"], gold, distractors)
    if "window" in strategies:
        units += build_window_units(lang, tier, group["qid"], gold, median_len)
    if "enriched" in strategies and group["query_text"]:
        units.append(build_enriched_unit(lang, tier, group["qid"], group["query_text"], gold))
    if "query" in strategies and group["query_text"]:
        units.append(build_query_unit(lang, tier, group["qid"], group["query_text"], gold))
    return units


## 5. Streaming and sampling (§5)

Handles both possible passage-container shapes discovered in Phase 0. For
the "single" (one row per query-passage pair) shape, rows for the same
query are not guaranteed contiguous after shuffling, so groups are
assembled with a bounded per-language buffer — documented best effort,
adjust if Phase 0 shows this shape and grouping looks wrong.


In [ ]:
def iter_query_groups(dataset_name, lang, seed, buffer_size, hf_token):
    ds = load_dataset(dataset_name, lang, split="train", streaming=True, token=(hf_token or None))
    ds = ds.shuffle(seed=seed, buffer_size=buffer_size)
    container = FIELD_MAP.get("passage_container", "list")
    seen_pairs = set()

    if container == "list":
        for src_row, row in enumerate(ds):
            qid = get_field(row, "query_id") or f"{lang}-row{src_row}"
            query_text = get_field(row, "query")
            passages = get_field(row, "passages") or []
            pids = get_field(row, "passage_id")
            gold_flags = get_field(row, "is_gold")
            plist = []
            for j, ptext in enumerate(passages):
                pid = pids[j] if isinstance(pids, (list, tuple)) and j < len(pids) else f"{qid}-p{j}"
                key = (qid, pid)
                if key in seen_pairs:
                    continue
                seen_pairs.add(key)
                is_gold = bool(gold_flags[j]) if isinstance(gold_flags, (list, tuple)) and j < len(gold_flags) else None
                plist.append({"pid": pid, "text": ptext, "is_gold": is_gold, "src_row": src_row})
            if plist:
                yield {"qid": qid, "query_text": query_text, "passages": plist}
    else:
        pending = {}
        FLUSH_AFTER = 6  # best-effort: assume a query's candidates arrive within this many sightings
        for src_row, row in enumerate(ds):
            qid = get_field(row, "query_id") or f"{lang}-row{src_row}"
            query_text = get_field(row, "query")
            ptext = get_field(row, "passages")
            pid = get_field(row, "passage_id") or f"row{src_row}"
            key = (qid, pid)
            if key in seen_pairs:
                continue
            seen_pairs.add(key)
            gold_raw = get_field(row, "is_gold")
            is_gold = bool(gold_raw) if gold_raw is not None else None
            g = pending.setdefault(qid, {"qid": qid, "query_text": query_text, "passages": []})
            g["passages"].append({"pid": pid, "text": ptext, "is_gold": is_gold, "src_row": src_row})
            if len(g["passages"]) >= FLUSH_AFTER:
                yield pending.pop(qid)
        for g in pending.values():
            yield g

def estimate_median_length(dataset_name, lang, seed, hf_token, sample_size=500):
    lens = []
    for group in itertools.islice(iter_query_groups(dataset_name, lang, seed, 2_000, hf_token), sample_size):
        for p in group["passages"]:
            lens.append(len(p["text"]))
    return float(np.median(lens)) if lens else 0.0


## 6. Embedding — Jina v3, task adapters, Matryoshka slicing (§6)

In [ ]:
from transformers import AutoModel

CORPUS_TASK = "retrieval.passage"   # §4.1-4.3
QUERY_TASK = "retrieval.query"      # §4.4 only

def load_jina_model(device="cuda" if torch.cuda.is_available() else "cpu"):
    model = AutoModel.from_pretrained(
        "jinaai/jina-embeddings-v3", trust_remote_code=True, torch_dtype=torch.float16,
    ).to(device).eval()
    revision = getattr(model.config, "_commit_hash", None) or "unknown"
    print(f"Loaded jinaai/jina-embeddings-v3 | revision={revision} | device={device}")
    return model, revision

@torch.no_grad()
def encode(model, texts, task, batch_size):
    assert task in (CORPUS_TASK, QUERY_TASK), f"bad task adapter: {task}"
    if not texts:
        return np.zeros((0, 1024), dtype=np.float32)
    out, i, bs = [], 0, batch_size
    pbar = tqdm(total=len(texts), desc=f"encode[{task}]")
    while i < len(texts):
        chunk = texts[i:i + bs]
        try:
            v = model.encode(chunk, task=task, truncate_dim=None)
            out.append(np.asarray(v, dtype=np.float32))
            i += bs
            pbar.update(len(chunk))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs <= 1:
                raise
            bs = max(1, bs // 2)
            print(f"OOM — halving batch size to {bs} and retrying")
    pbar.close()
    return np.vstack(out)

def embed_units(model, units, batch_size):
    """Query-strategy units get retrieval.query; everything else gets retrieval.passage (§6.2)."""
    corpus_idx = [i for i, u in enumerate(units) if u["payload"]["strategy"] != "query"]
    query_idx = [i for i, u in enumerate(units) if u["payload"]["strategy"] == "query"]
    v1024 = np.zeros((len(units), 1024), dtype=np.float32)
    if corpus_idx:
        v = encode(model, [units[i]["embed_text"] for i in corpus_idx], CORPUS_TASK, batch_size)
        for k, i in enumerate(corpus_idx):
            v1024[i] = v[k]
    if query_idx:
        v = encode(model, [units[i]["embed_text"] for i in query_idx], QUERY_TASK, batch_size)
        for k, i in enumerate(query_idx):
            v1024[i] = v[k]
    return v1024

def slice_all_dims(v1024_raw):
    return {d: matryoshka(v1024_raw, d) for d in [1024, 768, 512, 256, 128]}


### 6.4 Sparse vectors (BM25) with an Indic-tokenization check

In [ ]:
from fastembed import SparseTextEmbedding

_bm25 = None
def get_bm25():
    global _bm25
    if _bm25 is None:
        _bm25 = SparseTextEmbedding("Qdrant/bm25")
    return _bm25

TOKEN_RE = re.compile(r"\w+", re.UNICODE)

def sparse_fallback(text):
    toks = TOKEN_RE.findall(text.lower())
    counts = {}
    for t in toks:
        h = int(hashlib.md5(t.encode()).hexdigest()[:8], 16)
        counts[h] = counts.get(h, 0) + 1
    return list(counts.keys()), [float(v) for v in counts.values()]

USE_BM25_FALLBACK = False  # decided by the tokenization check below

def encode_sparse(texts):
    if USE_BM25_FALLBACK:
        pairs = [sparse_fallback(t) for t in texts]
    else:
        pairs = [(r.indices.tolist(), r.values.tolist()) for r in get_bm25().embed(texts)]
    return pairs

def check_indic_tokenization(sample_texts):
    global USE_BM25_FALLBACK
    pairs = [(r.indices.tolist(), r.values.tolist()) for r in get_bm25().embed(sample_texts)]
    mean_nonzero = float(np.mean([len(idx) for idx, _ in pairs])) if pairs else 0.0
    print(f"BM25 mean non-zero terms over {len(sample_texts)} samples: {mean_nonzero:.2f}")
    if mean_nonzero <= 1.0:
        print("Indic tokenization looks degenerate — falling back to regex tokenizer for sparse vectors.")
        USE_BM25_FALLBACK = True
    return mean_nonzero


## 7. Qdrant collection schema (§7)

In [ ]:
from qdrant_client import QdrantClient, models

CFG.validate()
client = QdrantClient(url=CFG.QDRANT_URL, api_key=CFG.QDRANT_API_KEY, timeout=120)
print("Qdrant client version check — collections:", [c.name for c in client.get_collections().collections])

def create_collection(client, cfg):
    client.recreate_collection(
        collection_name=cfg.COLLECTION_NAME,
        vectors_config={
            "dense_256": models.VectorParams(
                size=256, distance=models.Distance.COSINE, on_disk=True,
                hnsw_config=models.HnswConfigDiff(m=16, ef_construct=64),
            ),
            "dense_1024": models.VectorParams(
                size=1024, distance=models.Distance.COSINE, on_disk=True,
                hnsw_config=models.HnswConfigDiff(m=0),
            ),
        },
        sparse_vectors_config={
            "sparse_bm25": models.SparseVectorParams(
                index=models.SparseIndexParams(on_disk=False),
                modifier=models.Modifier.IDF,
            ),
        },
        quantization_config=models.ScalarQuantization(
            scalar=models.ScalarQuantizationConfig(
                type=models.ScalarType.INT8, quantile=0.99, always_ram=True,
            ),
        ),
        on_disk_payload=True,
        optimizers_config=models.OptimizersConfigDiff(
            default_segment_number=2, indexing_threshold=0,
        ),
    )
    for field, schema in [
        ("lang", models.PayloadSchemaType.KEYWORD),
        ("strategy", models.PayloadSchemaType.KEYWORD),
        ("tier", models.PayloadSchemaType.KEYWORD),
        ("qid", models.PayloadSchemaType.KEYWORD),
        ("parent_id", models.PayloadSchemaType.KEYWORD),
        ("is_gold", models.PayloadSchemaType.BOOL),
    ]:
        client.create_payload_index(cfg.COLLECTION_NAME, field_name=field, field_schema=schema, wait=True)
    print(f"Collection '{cfg.COLLECTION_NAME}' created with payload indexes.")

create_collection(client, CFG)


## 8. Upsert with checkpointing (§8)

In [ ]:
def ckpt_path(cfg):
    return os.path.join(cfg.DRIVE_DIR, "ckpt_msmarco.json")

def load_ckpt(cfg):
    p = ckpt_path(cfg)
    return json.load(open(p)) if os.path.exists(p) else {"lang": None, "done": 0}

def save_ckpt(cfg, lang, done):
    json.dump({"lang": lang, "done": done}, open(ckpt_path(cfg), "w"))

def make_point_structs(units, v256, v1024, sparse):
    points = []
    for j, u in enumerate(units):
        idx, val = sparse[j]
        points.append(models.PointStruct(
            id=point_id_for(u["uid"]),
            vector={
                "dense_256": v256[j].tolist(),
                "dense_1024": v1024[j].tolist(),
                "sparse_bm25": models.SparseVector(indices=idx, values=val),
            },
            payload={**u["payload"], "uid": u["uid"]},
        ))
    return points

def upsert_points(client, cfg, lang, points):
    state = load_ckpt(cfg)
    start = state["done"] if state["lang"] == lang else 0
    bs = cfg.UPSERT_BATCH_SIZE
    for i in tqdm(range(start, len(points), bs), desc=f"upsert[{lang}]"):
        batch = points[i:i + bs]
        for attempt in range(5):
            try:
                client.upsert(cfg.COLLECTION_NAME, points=batch, wait=False)
                break
            except Exception as e:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        save_ckpt(cfg, lang, i + bs)
    print(f"[{lang}] upserted {len(points)} points (resumed from {start})")


## 9. Orchestration — run the pipeline per language (§5-§8, resumable)

In [ ]:
def process_language(client, model, cfg, lang, tier_name, tier):
    print(f"=== Tier {tier_name} / {lang} ===")
    try:
        median_len = estimate_median_length(cfg.DATASET_NAME, lang, cfg.SEED, cfg.HF_TOKEN)
        units, sampled_qids, sample_query = [], [], None
        groups = itertools.islice(
            iter_query_groups(cfg.DATASET_NAME, lang, cfg.SEED, cfg.SHUFFLE_BUFFER, cfg.HF_TOKEN),
            tier["queries_per_lang"],
        )
        for group in groups:
            gu = build_units_for_group(group, lang, tier_name, tier["strategies"], median_len, tier["distractors"])
            units.extend(gu)
            sampled_qids.append(group["qid"])
            if sample_query is None and group["query_text"]:
                sample_query = group["query_text"]

        if not units:
            print(f"[{lang}] produced no units — skipping.")
            return None

        json.dump(sampled_qids, open(os.path.join(cfg.DRIVE_DIR, f"sampled_qids_{lang}.json"), "w"), ensure_ascii=False)

        v1024_raw = embed_units(model, units, cfg.EMBED_BATCH_SIZE)
        dims = slice_all_dims(v1024_raw)
        for d, arr in dims.items():
            np.save(os.path.join(cfg.DRIVE_DIR, f"emb_{lang}_{d}.npy"), arr)

        texts = [u["embed_text"] for u in units]
        sparse = encode_sparse(texts)

        points = make_point_structs(units, dims[256], dims[1024], sparse)
        upsert_points(client, cfg, lang, points)

        return {
            "lang": lang, "tier": tier_name, "n_points": len(points),
            "sample_query": sample_query, "median_len": median_len,
            "strategy_counts": {s: sum(1 for u in units if u["payload"]["strategy"] == s)
                                 for s in tier["strategies"]},
        }
    except Exception as e:
        print(f"[{lang}] FAILED: {e} — skipping language, continuing with the rest (§11).")
        return None

def run_pipeline(client, model, cfg, tiers_resolved):
    results = {}
    for tier_name, tier in tiers_resolved.items():
        for lang in tier["langs"]:
            r = process_language(client, model, cfg, lang, tier_name, tier)
            if r:
                results[lang] = r
    return results

# Load the model once, then run.
JINA_MODEL, JINA_REVISION = load_jina_model()

# --- entry point ---
# Run this once you're ready. Safe to re-run after a Colab disconnect: the
# per-language checkpoint on Drive resumes mid-upsert, and encode()/embed_units
# are re-derived from the same seeded stream (idempotent point IDs via uuid5).
RUN_RESULTS = run_pipeline(client, JINA_MODEL, CFG, TIERS_RESOLVED)


## 10. Build the index, then verify (§9)

In [ ]:
# Force flush, then build HNSW once instead of continuously during load.
client.upsert(CFG.COLLECTION_NAME, points=[], wait=True)
client.update_collection(CFG.COLLECTION_NAME,
                          optimizers_config=models.OptimizersConfigDiff(indexing_threshold=20_000))

while client.get_collection(CFG.COLLECTION_NAME).status != models.CollectionStatus.GREEN:
    print("waiting for collection to go GREEN...")
    time.sleep(10)
print("Collection GREEN.")


In [ ]:
def count_where(client, cfg, **filters):
    conditions = [models.FieldCondition(key=k, match=models.MatchValue(value=v)) for k, v in filters.items()]
    return client.count(cfg.COLLECTION_NAME, exact=True,
                         count_filter=models.Filter(must=conditions)).count

def run_acceptance_checks(client, cfg, tiers_resolved, run_results, model):
    all_points = sum(r["n_points"] for r in run_results.values())
    total_in_db = client.count(cfg.COLLECTION_NAME, exact=True).count
    assert total_in_db == all_points, f"count mismatch: db={total_in_db} expected={all_points}"
    print(f"[1] point count OK: {total_in_db}")

    for lang in tiers_resolved["A"]["langs"] + tiers_resolved["B"]["langs"]:
        for strat in ["atomic", "window", "enriched", "query"]:
            n = count_where(client, cfg, lang=lang, strategy=strat)
            assert n > 0, f"MISSING {lang}/{strat}"
    print("[2] tier A/B strategy coverage OK")

    gold = count_where(client, cfg, strategy="atomic", is_gold=True)
    tot = count_where(client, cfg, strategy="atomic")
    ratio = gold / tot if tot else 0
    assert 0.2 < ratio < 0.5, f"distractor ratio wrong: {gold}/{tot} = {ratio}"
    print(f"[3] distractor ratio OK: {ratio:.3f}")

    for lang, r in run_results.items():
        if not r.get("sample_query"):
            continue
        qv = matryoshka(encode(model, [r["sample_query"]], QUERY_TASK, 1), 256)[0]
        hits = client.query_points(
            cfg.COLLECTION_NAME, query=qv.tolist(), using="dense_256",
            limit=5, with_payload=True,
            query_filter=models.Filter(must=[models.FieldCondition(key="lang", match=models.MatchValue(value=lang))]),
        )
        assert hits.points and hits.points[0].score > 0.5, f"BAD RETRIEVAL {lang}"
        print(f"[4] {lang}: score={hits.points[0].score:.3f} text={hits.points[0].payload['text'][:80]!r}")

    sample_texts = []
    for lang, r in list(run_results.items())[:5]:
        sample_texts.append(r.get("sample_query") or "")
    mean_nonzero = check_indic_tokenization([t for t in sample_texts if t])
    assert mean_nonzero > 3, "BM25 tokenizer is not splitting Indic script"
    print(f"[5] BM25 alive: mean_nonzero={mean_nonzero:.2f}")

    print("All acceptance checks passed.")

run_acceptance_checks(client, CFG, TIERS_RESOLVED, RUN_RESULTS, JINA_MODEL)


## 11. Artifacts (§10)

In [ ]:
manifest = {
    "model": "jinaai/jina-embeddings-v3",
    "revision": JINA_REVISION,
    "corpus_task": CORPUS_TASK,
    "query_task": QUERY_TASK,
    "dims": [128, 256, 512, 768, 1024],
    "collection": CFG.COLLECTION_NAME,
    "dataset": CFG.DATASET_NAME,
    "timestamp": time.time(),
    "per_language": RUN_RESULTS,
    "tiers": {k: v["langs"] for k, v in TIERS_RESOLVED.items()},
    "projected_points": PROJECTED_POINTS,
}
json.dump(manifest, open(os.path.join(CFG.DRIVE_DIR, "manifest.json"), "w"), ensure_ascii=False, indent=2)
json.dump(FIELD_MAP, open(os.path.join(CFG.DRIVE_DIR, "field_map.json"), "w"), ensure_ascii=False, indent=2)

total_points = sum(r["n_points"] for r in RUN_RESULTS.values())
report_lines = [
    "# Ingest report", "",
    f"- Model: jinaai/jina-embeddings-v3 (revision {JINA_REVISION})",
    f"- Collection: {CFG.COLLECTION_NAME}",
    f"- Total points: {total_points}",
    "", "## Per-language", "",
]
for lang, r in RUN_RESULTS.items():
    report_lines.append(f"- **{lang}** (tier {r['tier']}): {r['n_points']} points — {r['strategy_counts']}")
open(os.path.join(CFG.DRIVE_DIR, "ingest_report.md"), "w", encoding="utf-8").write("\n".join(report_lines))

print("Artifacts written to", CFG.DRIVE_DIR)
print("- manifest.json, field_map.json, ingest_report.md")
print("- sampled_qids_{lang}.json, emb_{lang}_{dim}.npy per language")
